In [ ]:
# 配置文件 - 统一管理所有路径和参数
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import re
from intervaltree import IntervalTree
from aquarel import load_theme

# ============================
# 路径配置
# ============================
BASE_GWAS_DIR = '/mnt/d/幽门螺旋杆菌/Script/分析结果/GWAS/output/临床指标/'
BASE_FST_DIR = '/mnt/d/幽门螺旋杆菌/Script/分析结果/FST/output/临床指标'
GFF_FILE = '/mnt/d/幽门螺旋杆菌/Script/分析结果/FST/script/R方法/NC_000915.gff'
OUTPUT_BASE_DIR = '../output'

# ============================
# 分析目标配置
# ============================
ANALYSIS_TARGETS = {
    # 目标名称: (GWAS阈值, FST阈值1, FST阈值2, 输出格式, 是否启用注释)
    'Atrophyscore*-萎缩评分': {
        'gwas_threshold': 3,
        'fst_threshold1': 0.4,
        'fst_threshold2': 0.6,
        'output_format': 'pdf',
        'enable_gwas_annotation': True,
        'enable_fst_annotation': False,
        'max_annotations': 5
    },
    'Intestinalmetaplasiascore*-肠化生评分': {
        'gwas_threshold': 3,
        'fst_threshold1': 0.4,
        'fst_threshold2': 0.6,
        'output_format': 'pdf',
        'enable_gwas_annotation': True,
        'enable_fst_annotation': False,
        'max_annotations': 5
    },
    'AML-阿莫西林': {
        'gwas_threshold': 5,
        'fst_threshold1': 0.4,
        'fst_threshold2': 0.6,
        'output_format': 'tif',
        'enable_gwas_annotation': False,
        'enable_fst_annotation': False,
        'max_annotations': 5
    }
}

# ============================
# 绘图配置
# ============================
PLOT_CONFIG = {
    'figsize': (7, 7),
    'dpi': 300,
    'theme': 'boxy_light',
    'font_family': ['Arial'],
    'font_type': 42,
    'scatter_size': 8,
    'annotation_fontsize': 6,
    'colors': {
        'gwas_significant': '#FF5858',
        'gwas_normal': '#8A8A8A',
        'fst_high': '#AB5962',
        'fst_medium': '#71A48D',
        'fst_low': '#8A8A8A',
        'threshold_line': 'black'
    }
}

print("配置加载完成！")
print(f"分析目标: {list(ANALYSIS_TARGETS.keys())}")
print(f"基础路径: {BASE_GWAS_DIR}")

In [ ]:
# 数据加载和预处理函数
def load_and_merge_data(target_name):
    """
    加载GWAS和FST数据并合并
    
    Args:
        target_name (str): 分析目标名称
    
    Returns:
        pandas.DataFrame: 合并后的数据框
    """
    # 构建文件路径
    gwas_file = os.path.join(BASE_GWAS_DIR, target_name, 'bugwas_biallelic_lmmout_allSNPs.txt')
    fst_file = os.path.join(BASE_FST_DIR, target_name, '处理后FST.csv')
    
    # 检查文件是否存在
    if not os.path.exists(gwas_file):
        raise FileNotFoundError(f"GWAS文件不存在: {gwas_file}")
    if not os.path.exists(fst_file):
        raise FileNotFoundError(f"FST文件不存在: {fst_file}")
    
    # 读取数据
    df_gwas = pd.read_csv(gwas_file, sep='\t')
    df_gwas.rename(columns={'ps': 'Location'}, inplace=True)
    
    df_fst = pd.read_csv(fst_file)
    
    # 合并数据
    df_merged = df_fst.merge(df_gwas, on='Location', how='outer')
    df_merged = df_merged.loc[:, ['Location', 'Fst', 'negLog10']].fillna(0)
    
    return df_merged

def load_gff_annotations():
    """
    加载GFF注释文件并构建区间树
    
    Returns:
        IntervalTree: 包含基因注释的区间树
    """
    df_gff = pd.read_csv(GFF_FILE, sep='\t', header=None).loc[:, [3, 4, 8]]
    tree = IntervalTree()
    
    for _, row in df_gff.iterrows():
        # intervaltree 右端开区间，所以加 1
        tree.addi(row[3], row[4] + 1, row)
    
    return tree

def annotate_positions(df, tree):
    """
    为位点添加基因注释
    
    Args:
        df (pandas.DataFrame): 包含Location列的数据框
        tree (IntervalTree): 基因注释区间树
    
    Returns:
        pandas.DataFrame: 添加了annotation列的数据框
    """
    def annotate_row(pos):
        ivs = tree[pos]
        if ivs:
            # 任取一个（若存在重叠基因，可进一步筛选）
            gff_row = list(ivs)[0].data
            return gff_row[8]
        else:
            return None
    
    df_annotated = df.copy()
    df_annotated['annotation'] = df_annotated['Location'].apply(annotate_row)
    return df_annotated

print("数据处理函数定义完成！")

In [ ]:
# 绘图相关函数
def setup_plot_theme():
    """设置绘图主题和字体"""
    theme = load_theme(PLOT_CONFIG['theme'])
    theme.apply()
    
    plt.rcParams['font.family'] = PLOT_CONFIG['font_family']
    plt.rcParams['pdf.fonttype'] = PLOT_CONFIG['font_type']
    plt.rcParams['ps.fonttype'] = PLOT_CONFIG['font_type']
    
    return theme

def calculate_colors(gwas_data, config):
    """
    计算散点颜色
    
    Args:
        gwas_data (pandas.DataFrame): 包含negLog10和Fst列的数据
        config (dict): 当前目标的配置
    
    Returns:
        tuple: (gwas_colors, fst_colors)
    """
    neglog = gwas_data['negLog10'].values
    fst = gwas_data['Fst'].values
    
    colors_gwas = np.where(
        neglog > config['gwas_threshold'], 
        PLOT_CONFIG['colors']['gwas_significant'], 
        PLOT_CONFIG['colors']['gwas_normal']
    )
    
    colors_fst = np.where(
        fst > config['fst_threshold2'], 
        PLOT_CONFIG['colors']['fst_high'],
        np.where(
            fst > config['fst_threshold1'], 
            PLOT_CONFIG['colors']['fst_medium'], 
            PLOT_CONFIG['colors']['fst_low']
        )
    )
    
    return colors_gwas, colors_fst

def extract_gene_info(annotation_str):
    """
    从注释字符串中提取基因和产物信息
    
    Args:
        annotation_str (str): GFF注释字符串
    
    Returns:
        str or None: 简化的注释信息
    """
    if pd.isna(annotation_str) or annotation_str == 'None':
        return None
    
    pattern_prod = re.compile(r'product=([^;]+)')
    pattern_gene = re.compile(r'gene=([^;]+)')
    
    ann = str(annotation_str)
    prod = pattern_prod.search(ann)
    gene = pattern_gene.search(ann)
    
    parts = []
    if gene:
        parts.append(f"gene={gene.group(1)}")
    if prod:
        parts.append(f"product={prod.group(1)}")
    
    return "; ".join(parts) if parts else None

def add_annotations(ax, data, config, plot_type='gwas'):
    """
    为散点图添加注释
    
    Args:
        ax: matplotlib轴对象
        data (pandas.DataFrame): 数据
        config (dict): 配置参数
        plot_type (str): 'gwas' 或 'fst'
    """
    if plot_type == 'gwas' and not config['enable_gwas_annotation']:
        return
    if plot_type == 'fst' and not config['enable_fst_annotation']:
        return
    
    seen = set()
    
    if plot_type == 'gwas':
        threshold = config['gwas_threshold']
        value_col = 'negLog10'
        y_offset = 0.3
        y_max = data[value_col].max() * 1.05
    else:  # fst
        threshold = config['fst_threshold2']
        value_col = 'Fst'
        y_offset = 0.03
        y_max = 1.0
    
    # 筛选候选点
    candidates = data[data[value_col] > threshold].sort_values(value_col, ascending=False)
    
    for _, row in candidates.iterrows():
        short_ann = extract_gene_info(row['annotation'])
        if not short_ann or short_ann in seen:
            continue
        
        seen.add(short_ann)
        x = row['Location']
        y = row[value_col]
        y_text = min(y + y_offset, y_max)
        
        ax.annotate(
            short_ann,
            xy=(x, y), xytext=(x, y_text),
            arrowprops=dict(arrowstyle='->', color='black', lw=0.5),
            fontsize=PLOT_CONFIG['annotation_fontsize'], 
            ha='center', va='bottom', rotation=0,
            zorder=5
        )
        
        if len(seen) >= config['max_annotations']:
            break

print("绘图函数定义完成！")

In [ ]:
# 主要绘图函数
def create_manhattan_plot(gwas_data, target_name, config):
    """
    创建双轨曼哈顿图（GWAS + FST）
    
    Args:
        gwas_data (pandas.DataFrame): 包含Location, negLog10, Fst, annotation列的数据
        target_name (str): 分析目标名称
        config (dict): 当前目标的配置参数
    
    Returns:
        str: 输出文件路径
    """
    # 设置主题
    theme = setup_plot_theme()
    
    # 计算颜色
    colors_gwas, colors_fst = calculate_colors(gwas_data, config)
    
    # 创建图形
    fig, (ax_top, ax_bot) = plt.subplots(
        2, 1, sharex=True,
        figsize=PLOT_CONFIG['figsize'],
        gridspec_kw={'height_ratios': [1, 1]}
    )
    plt.subplots_adjust(hspace=0.05)
    
    # 上图：GWAS 曼哈顿图
    neglog = gwas_data['negLog10'].values
    ax_top.scatter(
        gwas_data['Location'], neglog,
        c=colors_gwas, s=PLOT_CONFIG['scatter_size'],
        rasterized=True, edgecolors='none'
    )
    ax_top.axhline(
        config['gwas_threshold'], 
        color=PLOT_CONFIG['colors']['threshold_line'], 
        linestyle='--', lw=1
    )
    ax_top.set_ylabel('-Log10(p-value)', fontsize=10)
    ax_top.set_ylim(0, neglog.max() * 1.05)
    ax_top.grid(False)
    ax_top.spines['bottom'].set_visible(False)
    ax_top.tick_params(bottom=False, labelbottom=False)
    
    # 添加GWAS注释
    add_annotations(ax_top, gwas_data, config, 'gwas')
    
    # 下图：FST 散点图
    fst = gwas_data['Fst'].values
    ax_bot.scatter(
        gwas_data['Location'], fst,
        c=colors_fst, s=PLOT_CONFIG['scatter_size'],
        rasterized=True, edgecolors='none'
    )
    ax_bot.axhline(
        config['fst_threshold1'], 
        color=PLOT_CONFIG['colors']['fst_medium'], 
        linestyle='--', lw=1
    )
    ax_bot.axhline(
        config['fst_threshold2'], 
        color=PLOT_CONFIG['colors']['fst_high'], 
        linestyle='--', lw=1
    )
    ax_bot.set_ylabel('Fst', fontsize=10)
    ax_bot.set_ylim(0, 1)
    ax_bot.grid(False)
    
    # 添加FST注释
    add_annotations(ax_bot, gwas_data, config, 'fst')
    
    # 设置X轴
    ax_bot.set_xlabel('Genomic position (bp)', fontsize=10)
    ax_bot.ticklabel_format(style='plain', axis='x')
    xmin, xmax = gwas_data['Location'].min(), gwas_data['Location'].max()
    ax_bot.set_xlim(xmin, xmax + 1000)
    
    # 保存图片
    output_dir = os.path.join(OUTPUT_BASE_DIR, target_name, 'func')
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, f'manhattan_twotrack_top10.{config["output_format"]}')
    
    theme.apply_transforms()
    plt.savefig(output_path, dpi=PLOT_CONFIG['dpi'], bbox_inches='tight')
    plt.show()
    
    return output_path

def process_single_target(target_name, config):
    """
    处理单个分析目标
    
    Args:
        target_name (str): 分析目标名称
        config (dict): 目标配置参数
    
    Returns:
        str: 输出文件路径
    """
    print(f"\\n正在处理: {target_name}")
    print(f"GWAS阈值: {config['gwas_threshold']}, FST阈值: {config['fst_threshold1']}/{config['fst_threshold2']}")
    
    try:
        # 加载数据
        df_merged = load_and_merge_data(target_name)
        print(f"数据加载完成，共 {len(df_merged)} 个位点")
        
        # 加载注释
        tree = load_gff_annotations()
        df_annotated = annotate_positions(df_merged, tree)
        print("基因注释完成")
        
        # 创建图片
        output_path = create_manhattan_plot(df_annotated, target_name, config)
        print(f"图片已保存到: {output_path}")
        
        return output_path
        
    except Exception as e:
        print(f"处理 {target_name} 时发生错误: {str(e)}")
        return None

print("主要处理函数定义完成！")

In [ ]:
# 主执行程序
def main():
    """主执行函数"""
    print("开始批量处理分析目标...")
    
    results = {}
    
    for target_name, config in ANALYSIS_TARGETS.items():
        output_path = process_single_target(target_name, config)
        results[target_name] = output_path
    
    # 输出结果汇总
    print("\\n" + "="*50)
    print("处理结果汇总:")
    print("="*50)
    
    for target_name, output_path in results.items():
        status = "成功" if output_path else "失败"
        print(f"{target_name}: {status}")
        if output_path:
            print(f"  -> {output_path}")
    
    return results

# 执行主程序
if __name__ == "__main__":
    results = main()

In [ ]:
# 单独处理示例（可选）
# 如果只想处理特定目标，可以使用以下代码：

def process_specific_target(target_name):
    """处理特定的分析目标"""
    if target_name not in ANALYSIS_TARGETS:
        print(f"错误：未找到目标 '{target_name}'")
        print(f"可用目标：{list(ANALYSIS_TARGETS.keys())}")
        return None
    
    config = ANALYSIS_TARGETS[target_name]
    return process_single_target(target_name, config)

# 示例：单独处理某个目标
# result = process_specific_target('AML-阿莫西林')

# 如果需要临时修改某个目标的配置，可以这样做：
def modify_target_config(target_name, **kwargs):
    """临时修改目标配置"""
    if target_name in ANALYSIS_TARGETS:
        original_config = ANALYSIS_TARGETS[target_name].copy()
        ANALYSIS_TARGETS[target_name].update(kwargs)
        print(f"已修改 {target_name} 的配置：{kwargs}")
        return original_config
    else:
        print(f"错误：未找到目标 '{target_name}'")
        return None

# 示例：临时提高阈值并启用注释
# modify_target_config('AML-阿莫西林', 
#                     gwas_threshold=6, 
#                     enable_gwas_annotation=True)

print("单独处理功能已准备就绪！")

# GWAS和FST双轨曼哈顿图可视化

## 功能说明
本notebook用于生成GWAS和FST的双轨曼哈顿图，包含以下主要功能：

1. **统一配置管理**：所有路径、参数、阈值在开头统一配置
2. **模块化函数**：数据加载、注释、绘图功能分别封装
3. **批量处理**：支持多个分析目标的批量处理
4. **灵活配置**：每个目标可单独设置阈值、输出格式、注释开关等

## 主要特点
- 🔧 **配置驱动**：通过`ANALYSIS_TARGETS`字典控制所有分析参数
- 📊 **双轨显示**：上图显示GWAS结果，下图显示FST结果
- 🏷️ **智能注释**：基于阈值自动标注显著基因/位点
- 🎨 **主题统一**：使用aquarel主题保持图片风格一致
- 📁 **路径管理**：所有文件路径统一管理，便于维护

## 输出说明
- 不同的分析目标使用不同的GWAS阈值（样本量大的采用更高阈值）
- 支持PDF和TIF等多种输出格式
- 可选择性开启/关闭基因注释功能